# Predicting Bird Geolocation Region from Latitude and Longitude

This notebook maps bird geolocations (latitude and longitude) to North American regions: US states, Canadian provinces, and other North American countries. It trains a classifier to predict the region and visualizes the results using folium.

## 1. Import Required Libraries
We use pandas, scikit-learn, folium, and geopandas for geospatial mapping and region assignment across North America.

In [16]:
import pandas as pd
import numpy as np
import folium
import geopandas as gpd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix
import matplotlib.pyplot as plt

## 2. Load and Inspect Data
Load the NABBP dataset including 'lat_dd' and 'lon_dd'. Birds may be located anywhere in North America (US, Canada, Mexico, Caribbean, Central America).

In [17]:

import os
from birds.settings import load_settings
settings = load_settings()
DATA_PATH = os.path.join(settings.nabbp_data_path, "NABBP_2025_grp_01.csv.gz")
df = pd.read_csv(DATA_PATH, compression='gzip', low_memory=True, engine='pyarrow')
df[['lat_dd', 'lon_dd']].head()

,lat_dd,lon_dd
0,58.25000,-116.41667
1,52.58333,-110.41667
2,53.41667,-110.91667
3,56.25000,-117.25000
4,51.25000,-111.75000


## Visualize Raw Geolocations with a Folium Map
Before preprocessing, let's visualize a sample of bird datapoints from the entire North American region on a map using folium.

In [18]:
# Plot a random sample of 5000 raw datapoints on a folium map and save to HTML
n_points = 5000
sample_df = df.sample(n=n_points, random_state=42) if len(df) > n_points else df
raw_map = folium.Map(location=[40, -100], zoom_start=3)  # Center of North America
for _, row in sample_df.iterrows():
    if pd.notnull(row['lat_dd']) and pd.notnull(row['lon_dd']):
        folium.CircleMarker(
            location=[row['lat_dd'], row['lon_dd']],
            radius=2,
            color='blue',
            fill=True,
            fill_color='blue',
            fill_opacity=0.5
        ).add_to(raw_map)
raw_map.save('north_america_raw_geolocations_map.html')
print('Map saved to north_america_raw_geolocations_map.html')

Map saved to north_america_raw_geolocations_map.html


## 3. Preprocess Geolocation Data
Filter out rows with missing or invalid latitude/longitude values and handle outliers. Use North American bounds for filtering.

In [19]:
# Filter for valid North American latitude and longitude, and remove null island (lat=0, lon=0)
na_lat_bounds = (7, 84)
na_lon_bounds = (-168, -52)
df = df.dropna(subset=['lat_dd', 'lon_dd'])
df = df[(df['lat_dd'] >= na_lat_bounds[0]) & (df['lat_dd'] <= na_lat_bounds[1])]
df = df[(df['lon_dd'] >= na_lon_bounds[0]) & (df['lon_dd'] <= na_lon_bounds[1])]
df = df[~((df['lat_dd'] == 0) & (df['lon_dd'] == 0))]
df[['lat_dd', 'lon_dd']].describe()

,lat_dd,lon_dd
count,1.812955e+06,1.812955e+06
mean,5.761840e+01,-9.910074e+01
std,1.313036e+01,2.069682e+01
min,8.416670e+00,-1.667765e+02
25%,5.150000e+01,-1.005000e+02
50%,6.050000e+01,-9.450000e+01
75%,6.750000e+01,-8.650000e+01
max,8.187439e+01,-5.425000e+01


## 4. Map Coordinates to North American Regions
Assign each (lat_dd, lon_dd) to a US state, Canadian province, or other North American country using a shapefile and geopandas.

In [20]:
north_america = {
    # Northern mainland countries
    "Canada": {"type": "country", "iso": "CAN"},
    "United States": {"type": "country", "iso": "USA"},
    "Mexico": {"type": "country", "iso": "MEX"},
    # Central America countries
    "Belize": {"type": "country", "iso": "BLZ"},
    "Costa Rica": {"type": "country", "iso": "CRI"},
    "El Salvador": {"type": "country", "iso": "SLV"},
    "Guatemala": {"type": "country", "iso": "GTM"},
    "Honduras": {"type": "country", "iso": "HND"},
    "Nicaragua": {"type": "country", "iso": "NIC"},
    "Panama": {"type": "country", "iso": "PAN"},
    # Caribbean countries
    "Antigua and Barbuda": {"type": "country", "iso": "ATG"},
    "Bahamas": {"type": "country", "iso": "BHS"},
    "Barbados": {"type": "country", "iso": "BRB"},
    "Cuba": {"type": "country", "iso": "CUB"},
    "Dominica": {"type": "country", "iso": "DMA"},
    "Dominican Republic": {"type": "country", "iso": "DOM"},
    "Grenada": {"type": "country", "iso": "GRD"},
    "Haiti": {"type": "country", "iso": "HTI"},
    "Jamaica": {"type": "country", "iso": "JAM"},
    "Saint Kitts and Nevis": {"type": "country", "iso": "KNA"},
    "Saint Lucia": {"type": "country", "iso": "LCA"},
    "Saint Vincent and the Grenadines": {"type": "country", "iso": "VCT"},
    "Trinidad and Tobago": {"type": "country", "iso": "TTO"},
    # Territories
    "Puerto Rico": {"type": "territory", "iso": "PRI"},
    "Greenland": {"type": "territory", "iso": "GRL"},
    "Bermuda": {"type": "territory", "iso": "BMU"},
    "Guadeloupe": {"type": "territory", "iso": "GLP"},
    "Martinique": {"type": "territory", "iso": "MTQ"},
    "Aruba": {"type": "territory", "iso": "ABW"},
    "Curaçao": {"type": "territory", "iso": "CUW"},
    "Sint Maarten": {"type": "territory", "iso": "SXM"},
    "Saint Martin": {"type": "territory", "iso": "MAF"},
    "Saint Barthélemy": {"type": "territory", "iso": "BLM"},
    "British Virgin Islands": {"type": "territory", "iso": "VGB"},
    "U.S. Virgin Islands": {"type": "territory", "iso": "VIR"},
    "Turks and Caicos Islands": {"type": "territory", "iso": "TCA"},
    "Cayman Islands": {"type": "territory", "iso": "CYM"},
    "Montserrat": {"type": "territory", "iso": "MSR"},
    "Anguilla": {"type": "territory", "iso": "AIA"},
    "Saint Pierre and Miquelon": {"type": "territory", "iso": "SPM"},
    "Bonaire, Sint Eustatius and Saba": {"type": "territory", "iso": "BES"},
    # Minor/disputed/uninhabited
    "Navassa Island": {"type": "territory", "iso": None},
    "Clipperton Island": {"type": "territory", "iso": None},
}

In [21]:
url = "https://naciscdn.org/naturalearth/110m/cultural/ne_110m_admin_0_countries.zip"
world = gpd.read_file(url)
world.head()

,featurecla,scalerank,LABELRANK,SOVEREIGNT,SOV_A3,ADM0_DIF,LEVEL,TYPE,TLC,ADMIN,...,FCLASS_TR,FCLASS_ID,FCLASS_PL,FCLASS_GR,FCLASS_IT,FCLASS_NL,FCLASS_SE,FCLASS_BD,FCLASS_UA,geometry
0,Admin-0 country,1,6,Fiji,FJI,0,2,Sovereign country,1,Fiji,...,None,None,None,None,None,None,None,None,None,"MULTIPOLYGON (((180 -16.06713, 180 -16.55522, ..."
1,Admin-0 country,1,3,United Republic of Tanzania,TZA,0,2,Sovereign country,1,United Republic of Tanzania,...,None,None,None,None,None,None,None,None,None,"POLYGON ((33.90371 -0.95, 34.07262 -1.05982, 3..."
2,Admin-0 country,1,7,Western Sahara,SAH,0,2,Indeterminate,1,Western Sahara,...,Unrecognized,Unrecognized,Unrecognized,None,None,Unrecognized,None,None,None,"POLYGON ((-8.66559 27.65643, -8.66512 27.58948..."
3,Admin-0 country,1,2,Canada,CAN,0,2,Sovereign country,1,Canada,...,None,None,None,None,None,None,None,None,None,"MULTIPOLYGON (((-122.84 49, -122.97421 49.0025..."
4,Admin-0 country,1,2,United States of America,US1,1,2,Country,1,United States of America,...,None,None,None,None,None,None,None,None,None,"MULTIPOLYGON (((-122.84 49, -120 49, -117.0312..."


In [25]:
# Assign country to each row in df based on lat_dd/lon_dd and world geometry
points_gdf = gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df['lon_dd'], df['lat_dd']), crs=world.crs)
country_name = "ADMIN"
joined = gpd.sjoin(points_gdf, world[['geometry', country_name]], how='left', predicate='within')
df['country_from_geom'] = joined[country_name].values
df['country_from_geom'].value_counts()

country_from_geom
Canada                      1111566
United States of America     408919
Mexico                         2975
Costa Rica                      371
Trinidad and Tobago             349
Cuba                             35
Panama                            5
Greenland                         2
Guatemala                         1
Name: count, dtype: int64

In [ ]:
# Use North American country/territory as the region label for prediction
X = df[['lat_dd', 'lon_dd']]
y = df['na_country']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train RandomForest classifier
clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)

,lat_dd,lon_dd,region
0,58.25000,-116.41667,North America Border
1,52.58333,-110.41667,North America Border
2,53.41667,-110.91667,North America Border
3,56.25000,-117.25000,North America Border
4,51.25000,-111.75000,North America Border


## 5. Train a Classification Model to Predict Region
Train a RandomForest classifier to predict the region (US state, Canadian province, or other North American country) from latitude and longitude.

In [ ]:
# Evaluate accuracy and show confusion matrix
acc = accuracy_score(y_test, y_pred)
print(f"Accuracy: {acc:.2f}")
cm = confusion_matrix(y_test, y_pred, labels=np.unique(y))
plt.figure(figsize=(16,10))
plt.imshow(cm, cmap='Blues')
plt.title('Confusion Matrix')
plt.xlabel('Predicted Country/Territory')
plt.ylabel('Actual Country/Territory')
plt.xticks(ticks=np.arange(len(np.unique(y))), labels=np.unique(y), rotation=90)
plt.yticks(ticks=np.arange(len(np.unique(y))), labels=np.unique(y))
plt.colorbar()
plt.show()

## 6. Evaluate Model Performance
Assess accuracy and display a confusion matrix for the region predictions.

In [ ]:
# Evaluate accuracy and show confusion matrix
acc = accuracy_score(y_test, y_pred)
print(f"Accuracy: {acc:.2f}")
cm = confusion_matrix(y_test, y_pred, labels=np.unique(y))
plt.figure(figsize=(16,10))
plt.imshow(cm, cmap='Blues')
plt.title('Confusion Matrix')
plt.xlabel('Predicted Region')
plt.ylabel('Actual Region')
plt.xticks(ticks=np.arange(len(np.unique(y))), labels=np.unique(y), rotation=90)
plt.yticks(ticks=np.arange(len(np.unique(y))), labels=np.unique(y))
plt.colorbar()
plt.show()

## 7. Visualize Predictions on a Folium Map
Plot a sample of predicted vs. actual regions on an interactive folium map for North America.

In [ ]:
# Visualize a sample of predictions on a folium map
sample = X_test.copy()
sample['actual_region'] = y_test.values
sample['predicted_region'] = y_pred
sample = sample.sample(n=200, random_state=42)

m = folium.Map(location=[40, -100], zoom_start=3)  # Center of North America
for _, row in sample.iterrows():
    color = 'green' if row['actual_region'] == row['predicted_region'] else 'red'
    folium.CircleMarker(
        location=[row['lat_dd'], row['lon_dd']],
        radius=4,
        color=color,
        fill=True,
        fill_color=color,
        popup=f"Actual: {row['actual_region']}, Predicted: {row['predicted_region']}"
    ).add_to(m)
m.save('north_america_predicted_vs_actual_map.html')
print('Map saved to north_america_predicted_vs_actual_map.html')